# EL-GHALI MOHAMED

### Q-LEARNING
Nous avons un **Agent** (une petite IA) qui évolue dans un **Environnement**. Il effectue des actions au hasard, et en fonction de la situation, il reçoit des **Récompenses** (positives ou négatives). Le but de l'algorithme est d'apprendre par essais/erreurs la meilleure stratégie pour maximiser ses récompenses.

Nous allons créer un petit labyrinthe (une grille 3x3). Le cerveau de notre agent s'appelle la **Q-Table** : c'est un tableau de triche qu'il remplit au fur et à mesure pour se souvenir de la "qualité" (Q-Value) de chaque action dans chaque case.

la formule du Q-Learning (Équation de Bellman) pour mettre à jour la Q-Table :


$$Q(S, A) = Q(S, A) + \alpha \times \left[ R + \gamma \times \max(Q(S', A')) - Q(S, A) \right]$$

* $S$ : État actuel / $A$ : Action choisie
* $R$ : Récompense obtenue
* $S'$ : Nouvel état après l'action
* $\alpha$ (Alpha) : Taux d'apprentissage (à quel point on remplace les anciennes connaissances)
* $\gamma$ (Gamma) : Facteur d'escompte (importance des récompenses futures)


#### Modèle Q-Learning (Apprentissage par Renforcement) "From S

Nous allons créer un jeu simple : une grille de 3x3 (9 cases numérotées de 0 à 8).

* L'agent commence à la case **0**.
* Il y a un piège à la case **5** (récompense de -10 et fin du jeu).
* Il y a un trésor à la case **8** (récompense de +10 et victoire !).
* Chaque pas dans le vide coûte de l'énergie (récompense de -1).

## 1. Création de l'Environnement (Le jeu)




In [1]:
import random

# Grille :
# [ 0 ][ 1 ][ 2 ]
# [ 3 ][ 4 ][ 5: Piège ]
# [ 6 ][ 7 ][ 8: Trésor ]

ACTIONS_POSSIBLES = ["Haut", "Bas", "Gauche", "Droite"]
ETATS = [0, 1, 2, 3, 4, 5, 6, 7, 8]
ETAT_INITIAL = 0
ETATS_TERMINAUX = [5, 8] # Le jeu s'arrête si on tombe dans le piège ou sur le trésor

def interagir_avec_environnement(etat_actuel, action):
    """Fonction qui applique les règles physiques de notre petit jeu."""
    nouvel_etat = etat_actuel

    #  Calcul de la nouvelle position
    if action == "Haut" and etat_actuel > 2:
        nouvel_etat -= 3
    elif action == "Bas" and etat_actuel < 6:
        nouvel_etat += 3
    elif action == "Gauche" and etat_actuel % 3 != 0:
        nouvel_etat -= 1
    elif action == "Droite" and etat_actuel % 3 != 2:
        nouvel_etat += 1

    # Calcul de la récompense
    if nouvel_etat == 8:
        recompense = 10  # Trésor
        fini = True
    elif nouvel_etat == 5:
        recompense = -10 # Piège
        fini = True
    elif nouvel_etat == etat_actuel:
        recompense = -2  # Pénalité pour s'être cogné dans un mur
        fini = False
    else:
        recompense = -1  # Pénalité pour chaque pas pour l'encourager à aller vite
        fini = False

    return nouvel_etat, recompense, fini

### Initialisation de la Q-Table

In [2]:
# Initialisation de la Q-Table avec des zéros
q_table = {}
for etat in ETATS:
    q_table[etat] = {}
    for action in ACTIONS_POSSIBLES:
        q_table[etat][action] = 0.0

print("Q-Table initiale créée. L'agent ne sait rien (tout est à 0).")
# Exemple pour la case départ : q_table[0] = {'Haut': 0.0, 'Bas': 0.0, 'Gauche': 0.0, 'Droite': 0.0}

Q-Table initiale créée. L'agent ne sait rien (tout est à 0).


 ### Algorithme du Q-Learning : Exploration et Mise à jour

In [3]:
def choisir_action(etat, q_table, epsilon):
    """L'agent choisit une action Exploration ou Exploitation."""
    # Exploration : On tire une action au hasard
    if random.uniform(0, 1) < epsilon:
        return random.choice(ACTIONS_POSSIBLES)

    # Exploitation : On prend la meilleure action connue dans cet état
    else:
        meilleure_action = None
        meilleure_valeur = float('-inf')
        for action in ACTIONS_POSSIBLES:
            valeur_actuelle = q_table[etat][action]
            if valeur_actuelle > meilleure_valeur:
                meilleure_valeur = valeur_actuelle
                meilleure_action = action
        return meilleure_action

def entrainer_agent(q_table, episodes=500, alpha=0.1, gamma=0.9, epsilon=0.2):
    """Fait jouer l'agent plusieurs centaines de fois pour remplir la Q-Table."""
    print("Début de l'entraînement...")

    for i in range(episodes):
        etat = ETAT_INITIAL
        fini = False

        while not fini:
            #  Choisir une action
            action = choisir_action(etat, q_table, epsilon)

            #  Effectuer l'action dans l'environnement
            nouvel_etat, recompense, fini = interagir_avec_environnement(etat, action)

            #  Calcul du maximum Q pour l'état SUIVANT (Anticipation)
            max_q_futur = max(q_table[nouvel_etat].values())

            #  Équation de Bellman
            # On met à jour la valeur de l'action qu'on vient de faire
            ancienne_valeur = q_table[etat][action]

            nouvelle_valeur = ancienne_valeur + alpha * (recompense + gamma * max_q_futur - ancienne_valeur)

            q_table[etat][action] = nouvelle_valeur

            # On se déplace mentalement dans la nouvelle case
            etat = nouvel_etat

    print(f"Entraînement terminé après {episodes} parties (épisodes) !")
    return q_table

### Entraînement et le test finale

In [4]:
# On entraîne l'agent (il va jouer 1000 fois au labyrinthe)
q_table_entrainee = entrainer_agent(q_table, episodes=1000, alpha=0.1, gamma=0.9, epsilon=0.3)

print("\n=== CERVEAU DE L'AGENT APRÈS ENTRAÎNEMENT (Q-Table partielle) ===")
# On affiche ce qu'il a appris pour la case Départ (0) et la case juste avant le trésor (7)
print(f"Case 0 (Départ) : {q_table_entrainee[0]}")
print(f"Case 7 (Avant Trésor) : {q_table_entrainee[7]}")

print("\n=== TEST EN CONDITIONS RÉELLES ===")
# On simule une partie où l'agent utilise 100% de ce qu'il a appris (epsilon = 0)
etat_actuel = ETAT_INITIAL
fini = False
chemin = [etat_actuel]
actions_prises = []

# Eviter les boucles infinies s'il a mal appris
tours_max = 10
tours = 0

while not fini and tours < tours_max:
    # On choisit la meilleure action sans hasard (epsilon = 0)
    action = choisir_action(etat_actuel, q_table_entrainee, epsilon=0.0)
    actions_prises.append(action)

    # On prend l'action
    nouvel_etat, recompense, fini = interagir_avec_environnement(etat_actuel, action)
    chemin.append(nouvel_etat)
    etat_actuel = nouvel_etat
    tours += 1

print(f"Actions choisies : {actions_prises}")
print(f"Cases visitées : {chemin}")

if etat_actuel == 8:
    print("-> L'AGENT A TROUVÉ LE TRÉSOR INTELLIGEMMENT !")
elif etat_actuel == 5:
    print("-> L'agent est tombé dans le piège...")
else:
    print("-> L'agent est perdu.")

Début de l'entraînement...
Entraînement terminé après 1000 parties (épisodes) !

=== CERVEAU DE L'AGENT APRÈS ENTRAÎNEMENT (Q-Table partielle) ===
Case 0 (Départ) : {'Haut': 2.121721804136398, 'Bas': 4.576962752435323, 'Gauche': 2.121935803335298, 'Droite': 4.579999999999986}
Case 7 (Avant Trésor) : {'Haut': 6.199576656852238, 'Bas': 6.993123892827443, 'Gauche': 6.186062841168576, 'Droite': 9.999999999999993}

=== TEST EN CONDITIONS RÉELLES ===
Actions choisies : ['Droite', 'Bas', 'Bas', 'Droite']
Cases visitées : [0, 1, 4, 7, 8]
-> L'AGENT A TROUVÉ LE TRÉSOR INTELLIGEMMENT !
